^# ABC Selective Attention — Structure Text Task

Updated to expose the new `confound_type` axis in evaluation slices and failure reporting.

In [1]:
import kaggle_benchmarks as kbench
import pandas as pd
import abc_selective_attention_utils as utils


In [2]:
from dataclasses import dataclass

@dataclass
class CountAnswer:
    count: int

@dataclass
class LineSelectionAnswer:
    lines: list[int]


In [3]:
DATASET_ROOT = utils.DEFAULT_DATASET_ROOT
CSV_PATH = DATASET_ROOT / "selective_attention/structure_text/structure_text_full.csv"

df = utils.load_csv(CSV_PATH)
print(CSV_PATH)
print("rows:", len(df))
print("columns:", sorted(df.columns.tolist()))


/kaggle/input/datasets/patrycjawegrzynowicz/abc-factorized-attention-benchmark/selective_attention/structure_text/structure_text_full.csv
rows: 680
columns: ['binding_distance', 'confound_count', 'confound_type', 'count_instruction', 'count_prompt', 'filter_instruction', 'filter_prompt', 'gold_count', 'gold_lines', 'line_length_noise', 'num_records', 'regime', 'regime_level', 'seed', 'serialization_style', 'structure_depth', 'structure_type', 'target_count', 'target_definition', 'text_input', 'unrelated_count']


In [4]:
base_cols = [
    "seed",
    "regime",
    "regime_level",
    "structure_type",
    "structure_depth",
    "binding_distance",
    "serialization_style",
    "num_records",
]

optional_cols = [
    c
    for c in ["target_count", "confound_count", "confound_type", "line_length_noise"]
    if c in df.columns
]

count_eval_df = df[base_cols + optional_cols + ["count_prompt", "gold_count"]].copy()
filter_eval_df = df[base_cols + optional_cols + ["filter_prompt", "gold_lines"]].copy()
groupings = utils.build_text_groupings(df)

print("count_eval_df columns:", count_eval_df.columns.tolist())
print("filter_eval_df columns:", filter_eval_df.columns.tolist())
print("groupings:", groupings)


count_eval_df columns: ['seed', 'regime', 'regime_level', 'structure_type', 'structure_depth', 'binding_distance', 'serialization_style', 'num_records', 'target_count', 'confound_count', 'confound_type', 'line_length_noise', 'count_prompt', 'gold_count']
filter_eval_df columns: ['seed', 'regime', 'regime_level', 'structure_type', 'structure_depth', 'binding_distance', 'serialization_style', 'num_records', 'target_count', 'confound_count', 'confound_type', 'line_length_noise', 'filter_prompt', 'gold_lines']
groupings: [['regime'], ['regime', 'regime_level'], ['structure_type'], ['num_records'], ['structure_depth'], ['structure_type', 'structure_depth'], ['binding_distance'], ['serialization_style'], ['target_count'], ['regime', 'target_count'], ['confound_count'], ['confound_type'], ['structure_type', 'confound_type'], ['line_length_noise']]


In [5]:
@kbench.task(store_task=False)
def single_structure_text_count_v1(
    llm,
    seed,
    regime,
    regime_level,
    structure_type,
    structure_depth,
    binding_distance,
    serialization_style,
    num_records,
    count_prompt,
    gold_count,
    target_count=None,
    confound_count=None,
    confound_type=None,
    line_length_noise=None,
) -> dict:
    response = llm.prompt(count_prompt, schema=CountAnswer)
    pred = int(response.count)
    gold = int(gold_count)

    result = {
        "task": "structure_text_count_v1",
        "model": llm.model,
        "seed": int(seed),
        "regime": regime,
        "regime_level": regime_level,
        "structure_type": structure_type,
        "structure_depth": structure_depth,
        "binding_distance": binding_distance,
        "serialization_style": serialization_style,
        "num_records": int(num_records),
        "gold_count": gold,
        "predicted_count": pred,
        "is_correct": pred == gold,
    }

    if target_count not in (None, ""):
        result["target_count"] = int(target_count)
    if confound_count not in (None, ""):
        result["confound_count"] = int(confound_count)
    if confound_type not in (None, ""):
        result["confound_type"] = confound_type
    if line_length_noise not in (None, ""):
        result["line_length_noise"] = int(line_length_noise)

    return result


In [6]:
@kbench.task(store_task=False)
def single_structure_text_filter_v1(
    llm,
    seed,
    regime,
    regime_level,
    structure_type,
    structure_depth,
    binding_distance,
    serialization_style,
    num_records,
    filter_prompt,
    gold_lines,
    target_count=None,
    confound_count=None,
    confound_type=None,
    line_length_noise=None,
) -> dict:
    gold = utils.parse_gold_lines(gold_lines) if isinstance(gold_lines, str) else [int(x) for x in gold_lines]

    pred = None
    error = None
    error_type = None

    try:
        response = llm.prompt(filter_prompt, schema=LineSelectionAnswer)
        pred = [int(x) for x in response.lines]
    except Exception as exc:
        error_type, error = utils.classify_prompt_error(exc)

    is_correct = pred == gold
    has_error = error_type is not None
    is_failure = not is_correct
    failure_type = "ok" if is_correct else (error_type if has_error else "wrong_answer")

    kbench.assertions.assert_true(
        is_correct,
        expectation=f"Expected {gold}, got {pred}" + (f" | error_type={error_type} | error={error}" if error else ""),
    )

    result = {
        "task": "structure_text_filter_v1",
        "model": llm.model,
        "seed": int(seed),
        "regime": regime,
        "regime_level": regime_level,
        "structure_type": structure_type,
        "structure_depth": structure_depth,
        "binding_distance": binding_distance,
        "serialization_style": serialization_style,
        "num_records": int(num_records),
        "gold_lines": gold,
        "predicted_lines": pred,
        "is_correct": is_correct,
        "is_failure": is_failure,
        "has_error": has_error,
        "failure_type": failure_type,
    }

    if target_count not in (None, ""):
        result["target_count"] = int(target_count)
    if confound_count not in (None, ""):
        result["confound_count"] = int(confound_count)
    if confound_type not in (None, ""):
        result["confound_type"] = confound_type
    if line_length_noise not in (None, ""):
        result["line_length_noise"] = int(line_length_noise)
    if error_type is not None:
        result["error_type"] = error_type
    if error is not None:
        result["error"] = error

    return result


In [10]:
SELECTED_TASK = "filter"  # "count" or "filter"
N_JOBS = 8

In [8]:
@kbench.task(store_task=False)
def structure_text_count_dataset(llm) -> float:
    flat, summary = utils.run_dataset_single_model(
        item_task=single_structure_text_count_v1,
        llm=llm,
        eval_df=count_eval_df,
        task_name="structure_text_count_dataset",
        n_jobs=N_JOBS,
        groupings=groupings,
    )
    print("=== failures: structure_text_count_dataset ===")
    utils.print_failures(
        flat,
        columns=utils.available_columns(
            flat,
            [
                "seed",
                "regime",
                "regime_level",
                "structure_type",
                "structure_depth",
                "binding_distance",
                "serialization_style",
                "target_count",
                "confound_count",
                "confound_type",
                "line_length_noise",
                "num_records",
                "gold_count",
                "predicted_count",
            ],
        ),
    )
    return summary["accuracy"] * 100.0


@kbench.task(store_task=False)
def structure_text_filter_dataset(llm) -> float:
    flat, summary = utils.run_dataset_single_model(
        item_task=single_structure_text_filter_v1,
        llm=llm,
        eval_df=filter_eval_df,
        task_name="structure_text_filter_dataset",
        n_jobs=N_JOBS,
        groupings=groupings,
    )
    print("=== failures: structure_text_filter_dataset ===")
    utils.print_failures(
        flat,
        columns=utils.available_columns(
            flat,
            [
                "seed",
                "regime",
                "regime_level",
                "structure_type",
                "structure_depth",
                "binding_distance",
                "serialization_style",
                "target_count",
                "confound_count",
                "confound_type",
                "line_length_noise",
                "num_records",
                "gold_lines",
                "predicted_lines",
                "failure_type",
                "error_type",
                "error",
            ],
        ),
    )
    return summary["accuracy"] * 100.0


In [9]:
if SELECTED_TASK == "count":
    run = structure_text_count_dataset.run(kbench.llm)
elif SELECTED_TASK == "filter":
    run = structure_text_filter_dataset.run(kbench.llm)
else:
    raise ValueError(f"Unknown SELECTED_TASK: {SELECTED_TASK}")
run


[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done  20 tasks      | elapsed:   15.7s
[Parallel(n_jobs=6)]: Done  60 tasks      | elapsed:   53.2s
[Parallel(n_jobs=6)]: Done 116 tasks      | elapsed:  1.8min
[Parallel(n_jobs=6)]: Done 188 tasks      | elapsed:  3.0min
[Parallel(n_jobs=6)]: Done 276 tasks      | elapsed:  4.6min
[Parallel(n_jobs=6)]: Done 380 tasks      | elapsed:  6.4min
[Parallel(n_jobs=6)]: Done 500 tasks      | elapsed:  9.3min
[Parallel(n_jobs=6)]: Done 636 tasks      | elapsed: 12.0min
[Parallel(n_jobs=6)]: Done 680 out of 680 | elapsed: 13.1min finished


structure_text_filter_dataset: 679/680 = 99.85%
=== structure_text_filter_dataset: by regime ===
                        regime  passed  total  accuracy
            baseline_structure      20     20    1.0000
        binding_distance_sweep      60     60    1.0000
                      combined      60     60    1.0000
                confound_sweep      80     80    1.0000
           confound_type_sweep     260    260    1.0000
     serialization_style_sweep      60     60    1.0000
         structure_depth_sweep      60     60    1.0000
target_count_x_structure_depth      79     80    0.9875
=== structure_text_filter_dataset: by regime, regime_level ===
                        regime                   regime_level  passed  total  accuracy
            baseline_structure                       baseline      20     20      1.00
        binding_distance_sweep                            far      20     20      1.00
        binding_distance_sweep                         medium      20     2

BokehModel(combine_events=True, render_bundle={'docs_json': {'4e8ffda7-12be-49f3-b49b-8989ec788c49': {'version…

In [ ]:
%choose structure_text_filter_dataset